In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings, random, os
warnings.filterwarnings("ignore")

# 1. CONFIG & REPRODUCIBILITY
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 2. LOAD DATA
DATA_PATH = '/kaggle/input/datasets/jpcnvn/downsampled-data/al6011_downsampled_full.xlsx'

xf = pd.ExcelFile(DATA_PATH)
sheets = [s for s in xf.sheet_names if s != 'Summary']

dfs = []
for s in sheets:
    df = pd.read_excel(DATA_PATH, sheet_name=s)
    df['condition'] = s
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(f"Total: {len(data)} points | Conditions: {data['condition'].nunique()}")
print(f"Columns: {data.columns.tolist()}")

# 3. TRAIN / VAL / TEST SPLIT — Stratified within each condition
# Within each (T, ε̇) condition: 70% train / 15% val / 15% test
# Every condition appears in all splits → fair evaluation

feature_cols = ['T_K', 'ln_sr', 'eps_true']
target_col = 'sigma_true'

X = data[feature_cols].values.astype(np.float32)
y = data[target_col].values.astype(np.float32).reshape(-1, 1)

train_idx, val_idx, test_idx = [], [], []
for cond in data['condition'].unique():
    idx = data[data['condition'] == cond].index.values.copy()
    np.random.shuffle(idx)
    n = len(idx)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_idx.extend(idx[:n_train])
    val_idx.extend(idx[n_train:n_train + n_val])
    test_idx.extend(idx[n_train + n_val:])

print(f"Split: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")

# 4. SCALING
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_X.fit_transform(X[train_idx])
X_val_s   = scaler_X.transform(X[val_idx])
X_test_s  = scaler_X.transform(X[test_idx])

y_train_s = scaler_y.fit_transform(y[train_idx])
y_val_s   = scaler_y.transform(y[val_idx])
y_test_s  = scaler_y.transform(y[test_idx])

# 5. DATALOADERS
BATCH_SIZE = 64

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_s, y_train_s, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val_s,   256, shuffle=False)
test_loader  = make_loader(X_test_s,  y_test_s,  256, shuffle=False)

# 6. MODEL
class FlowStressANN(nn.Module):
    """
    Black-box MLP: [T_K, ln(ε̇), ε] → σ
    No physics embedded. Model #1 baseline.
    """
    def __init__(self, input_dim=3, hidden_dims=[128, 128, 64], dropout=0.1):
        super().__init__()
        layers = []
        in_d = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(in_d, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = FlowStressANN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {n_params:,}")
print(model)

# 7. TRAINING
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=30, min_lr=1e-6
)

EPOCHS = 500
PATIENCE = 60

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        pred = model(X_b)
        loss = criterion(pred, y_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_b)
    train_loss = epoch_loss / len(train_idx)

    # Validate
    model.eval()
    val_loss_sum = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            val_loss_sum += criterion(model(X_b), y_b).item() * len(X_b)
    val_loss = val_loss_sum / len(val_idx)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), 'best_ann_model.pt')
    else:
        patience_counter += 1

    if (epoch + 1) % 50 == 0 or epoch == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:4d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr:.2e}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}. Best: {best_epoch+1}")
        break

print(f"\nBest val loss: {best_val_loss:.6f} at epoch {best_epoch+1}")

# 8. EVALUATION
model.load_state_dict(torch.load('best_ann_model.pt', weights_only=True))
model.eval()

def predict_from_scaled(X_scaled):
    """Predict σ (MPa) from pre-scaled inputs. No shuffle → correct alignment."""
    X_t = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        y_pred_s = model(X_t).cpu().numpy()
    return scaler_y.inverse_transform(y_pred_s)

y_train_pred = predict_from_scaled(X_train_s)
y_val_pred   = predict_from_scaled(X_val_s)
y_test_pred  = predict_from_scaled(X_test_s)

y_train_real = y[train_idx]
y_val_real   = y[val_idx]
y_test_real  = y[test_idx]

def compute_metrics(y_true, y_pred, label=""):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # AARE: filter σ > 5 MPa to avoid inflation from near-zero stress
    mask = y_true.flatten() > 5.0
    aare = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"{label:12s} | R² = {r2:.6f} | RMSE = {rmse:.4f} MPa | AARE = {aare:.2f}%")
    return r2, rmse, aare

print("\n" + "="*65)
print("EVALUATION (original MPa space)")
print("="*65)
r2_tr, rmse_tr, aare_tr = compute_metrics(y_train_real, y_train_pred, "Train")
r2_va, rmse_va, aare_va = compute_metrics(y_val_real,   y_val_pred,   "Validation")
r2_te, rmse_te, aare_te = compute_metrics(y_test_real,  y_test_pred,  "Test")

# 9. PLOTS

# 9a. Training curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train', alpha=0.8)
ax.plot(val_losses, label='Validation', alpha=0.8)
ax.axvline(best_epoch, color='r', ls='--', alpha=0.5, label=f'Best ({best_epoch+1})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (scaled)')
ax.set_title('Training & Validation Loss')
ax.legend(); ax.set_yscale('log')
plt.tight_layout()
plt.savefig('ann_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# 9b. Scatter: predicted vs experimental
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, yt, yp, lbl in zip(axes,
    [y_train_real, y_val_real, y_test_real],
    [y_train_pred, y_val_pred, y_test_pred],
    ['Train', 'Validation', 'Test']):
    ax.scatter(yt, yp, alpha=0.5, s=15, edgecolors='none')
    lims = [0, max(yt.max(), yp.max()) * 1.05]
    ax.plot(lims, lims, 'r--', lw=1.5, label='y = x')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Experimental σ (MPa)'); ax.set_ylabel('Predicted σ (MPa)')
    ax.set_title(f'{lbl}'); ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('ann_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# 9c. Flow stress curves — predict on ALL data for complete visualization
X_all_s = scaler_X.transform(X)
y_all_pred = predict_from_scaled(X_all_s)
data['sigma_pred'] = y_all_pred.flatten()

temperatures = sorted(data['T_C'].unique())
strain_rates = sorted(data['strain_rate'].unique())
markers = ['o', 's', '^']
colors_sr = ['#e41a1c', '#377eb8', '#4daf4a']

n_temps = len(temperatures)
ncols = 4
nrows = (n_temps + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes = axes.flatten()

for i, T in enumerate(temperatures):
    ax = axes[i]
    for j, sr in enumerate(strain_rates):
        sub = data[(data['T_C'] == T) & (data['strain_rate'] == sr)].sort_values('eps_true')
        if len(sub) == 0:
            continue
        ax.scatter(sub['eps_true'], sub['sigma_true'], marker=markers[j],
                   color=colors_sr[j], s=20, alpha=0.6, label=f'{sr} s⁻¹ (exp)')
        ax.plot(sub['eps_true'], sub['sigma_pred'], color=colors_sr[j], lw=1.5)
    ax.set_title(f'T = {T}°C')
    ax.set_xlabel('True Strain')
    ax.set_ylabel('True Stress (MPa)')
    if i == 0:
        ax.legend(fontsize=7)

for k in range(n_temps, len(axes)):
    fig.delaxes(axes[k])

plt.suptitle('ANN: Predicted (lines) vs Experimental (markers)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('ann_flow_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# 10. SUMMARY
print("\n" + "="*65)
print("SUMMARY — ANN (Black-Box MLP)")
print("="*65)
print(f"Architecture: MLP [3 → 128 → 128 → 64 → 1], ReLU, Dropout=0.1")
print(f"Parameters:   {n_params:,}")
print(f"Data:         {len(data)} pts, 21 conditions, Δε = 0.005")
print(f"Split:        {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"Best epoch:   {best_epoch + 1}")
print(f"")
print(f"{'Metric':<12} {'Train':>10} {'Val':>10} {'Test':>10}")
print(f"{'R²':<12} {r2_tr:>10.6f} {r2_va:>10.6f} {r2_te:>10.6f}")
print(f"{'RMSE (MPa)':<12} {rmse_tr:>10.4f} {rmse_va:>10.4f} {rmse_te:>10.4f}")
print(f"{'AARE (%)':<12} {aare_tr:>10.2f} {aare_va:>10.2f} {aare_te:>10.2f}")

# 11. SAVE PREDICTIONS
results = data.copy()
results['split'] = 'N/A'
for idx_list, label in [(train_idx, 'train'), (val_idx, 'val'), (test_idx, 'test')]:
    results.loc[idx_list, 'split'] = label
results['residual'] = results['sigma_true'] - results['sigma_pred']
results.to_csv('ann_predictions.csv', index=False)
print("\nSaved: ann_predictions.csv")